In [0]:
import json, os, sys
from datetime import datetime, timezone
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common, refresh

# refresh/01_refresh_out: SOURCE half of a controlled refresh (5.11, src/lh/refresh.py). Started only by the refresh
# workflow after approval, as the source environment's job account (prod -> test: sp_<code>_prod; test -> dev:
# sp_<code>_test). Writes a sanitised sample into the TARGET's ops.refresh_in volume: real values never leave the
# source. Prints table names and row counts only.
dbutils.widgets.dropdown("target", "test", ["test", "dev"])
target = dbutils.widgets.get("target")
rcfg, parents, ids = refresh.load()
source, pct = refresh.flow(rcfg, target)
cfg, tcfg = common.load_config(source), common.load_config(target)
if not (cfg["refresh_out"] and tcfg["refresh_in"]):
    raise ValueError(f"refresh {source} -> {target} is not allowed by config/{source}.yml and config/{target}.yml")
key = dbutils.secrets.get("lh_refresh", "refresh_key")       # put there by the refresh workflow, never printed
run = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
base = f"/Volumes/{tcfg['catalog']}/ops/refresh_in"
counts = refresh.out(spark, cfg["catalog"], f"{base}/{run}", key, pct, rcfg, parents, ids)
dbutils.fs.put(f"{base}/{run}/_manifest.json", refresh.manifest(source, target, pct, counts), overwrite=True)
dbutils.fs.put(f"{base}/_latest", run, overwrite=True)                # the target loads this run
for t, n in counts.items():
    print(f"{t:22} {n:>8} rows")
common.log_run(spark, cfg, "refresh_out", "success", sum(counts.values()), f"to {target}, run {run}, {pct}% of people")
dbutils.notebook.exit(json.dumps({"run": run, "rows": sum(counts.values())}))